In [1]:
import numpy as np
from tqdm import tqdm
from games.kuhn import KuhnPoker
from agents.counterfactualregret_t import CounterFactualRegret
from agents.ismcts import ISMCTS
from agents.minimax import MiniMax
from agents.agent_random import RandomAgent

In [2]:
g = KuhnPoker(initial_player=0)

In [3]:
my_agents = {
    g.agents[0]: CounterFactualRegret(game=g, agent=g.agents[0]),
    #g.agents[0]: RandomAgent(game=g, agent=g.agents[1]),
    #g.agents[1]: ISMCTS(game=g, agent=g.agents[1], simulations=200, rollouts=10, rollout_iterations=100),
    g.agents[1]: CounterFactualRegret(game=g, agent=g.agents[1])
}

In [4]:
g.reset()
while not g.done():
    g.render()
    print(f"Agent {g.agent_selection}")
    action = my_agents[g.agent_selection].action()
    print(f"Action {action} - move {g.action_move(action)}")
    g.step(action)
g.render()
for agent in g.agents:
    print(f"Reward {agent} = {g.reward(agent)}")

agent_0 J 
agent_1 K 
Agent agent_0
Node 0 does not exist. Playing random.
Action 0 - move p
agent_0 J p
agent_1 K p
Agent agent_1
Node 2p does not exist. Playing random.
Action 1 - move b
agent_0 J pb
agent_1 K pb
Agent agent_0
Node 0pb does not exist. Playing random.
Action 1 - move b
agent_0 J pbb
agent_1 K pbb
Reward agent_0 = -2
Reward agent_1 = 2


In [5]:
train_iterations = 1000

for agent in g.agents:
    print('Training agent ' + agent)
    if hasattr(my_agents[agent], 'train'):
        my_agents[agent].train(train_iterations)
        print(dict(map(lambda n: (n, np.round(my_agents[agent].node_dict[n].policy(), 3)), my_agents[agent].node_dict.keys())))

Training agent agent_0


100%|██████████| 1000/1000 [00:04<00:00, 231.22it/s]


{'0': array([0.778, 0.222]), '1p': array([0.998, 0.002]), '0pb': array([1., 0.]), '1b': array([0.586, 0.414]), '1': array([0.984, 0.016]), '2p': array([0., 1.]), '1pb': array([0.431, 0.569]), '2b': array([0., 1.]), '0p': array([0.644, 0.356]), '0b': array([1., 0.]), '2': array([0.319, 0.681]), '2pb': array([0., 1.])}
Training agent agent_1


100%|██████████| 1000/1000 [00:04<00:00, 236.12it/s]

{'1': array([0.93, 0.07]), '0p': array([0.696, 0.304]), '1pb': array([0.356, 0.644]), '0b': array([1., 0.]), '2': array([0.247, 0.753]), '2pb': array([0., 1.]), '1p': array([0.999, 0.001]), '1b': array([0.622, 0.378]), '2p': array([0., 1.]), '2b': array([0., 1.]), '0': array([0.793, 0.207]), '0pb': array([1., 0.])}


In [27]:
cum_rewards = dict(map(lambda agent: (agent, 0.), g.agents))
niter = 10000
for _ in tqdm(range(niter)):
    g.reset()
    #g.render()
    turn = 0
    while not g.done():
        #print('Turn: ', turn)
        #print('\tPlayer: ', g.agent_selection)
        #print('\tObservation: ', g.observe(g.agent_selection))
        a = my_agents[g.agent_selection].action()
        #print('\tAction: ', g._moves[a])
        g.step(action=a)
        turn += 1
    #print('Rewards: ', g.rewards)
    for agent in g.agents:
        cum_rewards[agent] += g.rewards[agent]
print('Average rewards:', dict(map(lambda agent: (agent, cum_rewards[agent]/niter), g.agents)))


100%|██████████| 10000/10000 [00:02<00:00, 4575.15it/s]

Average rewards: {'agent_0': -0.0697, 'agent_1': 0.0697}
